# Parallel computing
### Ex. 10.1: Parallelizing the genetic algorithm in *ex.9* 
> ***goal**: use the **MPI** library to perform multiple parallel* GA *searches with periodical exchange of best-fitted individuals*

To write the parallel verion of the code in ex. 9 we add a `MpiEnv` class (automatically calls `MPI_Initialize()` in the constructor and `MPI_Finalize()` in the destructor), along with some useful functions that wrap communication and comparison methods:
```cpp
class MpiEnv {
private:
  int _rank = 0, _size = 1;

public:
  MpiEnv(int &argc, char **&argv);
  ~MpiEnv();
  int rank() const, size() const;
};

std::vector<City> generate_and_bcast_cities(std::size_t, std::size_t, Random &,
                                            const MpiEnv &);
int rank_of_best(double local_best, const MpiEnv &);
````

In addition, we can write a new constructor for the `Random` class
```cpp
Random::Random(int);
```

that sets a different seed for the different ranks, avoiding identical parallel runs.

Note that `generate_and_bcast_cities(...)` is defined as
```cpp
std::vector<City> generate_and_bcast_cities(std::size_t ndim,
                                            std::size_t ncities, Random &rnd,
                                            const MpiEnv &mpi) {
  std::vector<City> cities{};
  std::vector<double> flat_coord{};

  if (mpi.rank() == 0) {
    cities = cities_in_a_square(ncities, rnd);
    for (auto &c : cities) {
      for (auto x : c.get_pos()) {
        flat_coord.emplace_back(x);
      }
    }
  } else {
    flat_coord.resize(ndim * ncities);
  }
  MPI_Bcast(flat_coord.data(), ndim * ncities, MPI_DOUBLE, 0, MPI_COMM_WORLD);

  if (mpi.rank() == 0) {
    return cities;
  }

  for (std::size_t c = 0; c < ncities; c++) {
    std::string name = "city" + std::to_string(c);
    std::vector<double> pos{};
    for (std::size_t i = 0; i < ndim; i++) {
      pos.emplace_back(flat_coord[c * ndim + i]);
    }
    cities.emplace_back(name, pos);
  }

  return cities;
}
```

this allows the cities to be randomly generated only by the $\text{rank} = 0$ process who then broadcasts them to the others, otherwise different `Random` seeds would produce different cities for each process.

Lastly, to implement migrations, we can set a `std::size_t nmigr` parameter and define
```cpp
void migrate(Population &pop, const TSP &tsp, Random &rnd, const MpiEnv &mpi) {
  const int rank = mpi.rank(), size = mpi.size();
  if (size == 1) {
    return;
  }

  std::vector<int> pop_order(size);
  if (rank == 0) {
    std::vector<std::size_t> pop_order_sizet = rnd_order(size, rnd);
    pop_order = std::vector<int>(
        pop_order_sizet.begin(),
        pop_order_sizet.end()); /* range constructor: converts each element (ok
                                   for sending via MPI) */
  }

  MPI_Bcast(pop_order.data(), size, MPI_INT, 0, MPI_COMM_WORLD);

  const int pos_in_pop_order =
      std::find(pop_order.begin(), pop_order.end(), rank) - pop_order.begin();
  const int destination_rank = pop_order[(pos_in_pop_order + 1) % size];
  const int source_rank = pop_order[(pos_in_pop_order - 1 + size) % size];

  const std::vector<std::size_t> &best_order = pop.best().order();
  std::vector<int> buffer(best_order.begin(), best_order.end());

  MPI_Sendrecv_replace(buffer.data(), buffer.size(), MPI_INT, destination_rank,
                       0, source_rank, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

  pop.receive_migrant(std::vector<std::size_t>(buffer.begin(), buffer.end()),
                      tsp);
}
```

What this function does is
1. define a random order of ranks, so that each process sends its best individual to the *randomly-chosen* next one, and receives from the previous one
2. use `MPI_Sendrecv_replace` instead of single `MPI_Send`/`MPI_Recv` to avoid deadlocks: this simultaneously sends and receives data to/from the same buffer, effectively replacing its content in-place

*(note that it also avoids the complications in manually managing non-blocking `MPI_Isend`/`MPI_Irecv`)*

3. after exchanging data between processes, the function `Population::receive_migrant(std::vector<std::size_t>, const TSP&)` simply rebuilds the received `Route`, overwriting the current worst one

With these additions, we rewrite the `main()` function as
```cpp
int main(int argc, char **argv) {
  /* environment objects */
  MpiEnv mpi(argc, argv);
  Random rnd(mpi.rank());
  /* tsp objects */
  std::size_t ndim = 2, ncities = 34, nroutes = 500, ngens = 1000, nmigr = 30;
  std::vector<City> cities = generate_and_bcast_cities(ndim, ncities, rnd, mpi);
  TSP tsp(cities);
  Population pop(nroutes, tsp, rnd);
  /* output objects */
  std::string output_dir = "output";
  std::string hist_filename =
      output_dir + "/history_rank_" + std::to_string(mpi.rank()) + ".log";
  std::string best_filename = output_dir + "/best_all_ranks.log";

  /* run optimization and print history */
  std::ofstream hist(hist_filename);
  hist << "GEN,BEST,BEST_HALF_AVG\n"
       << 0 << "," << pop.best_length() << "," << pop.half_mean_length()
       << "\n";

  for (std::size_t gen = 0; gen < ngens; gen++) {
    pop.next_generation(tsp, rnd);
    if ((gen + 1) % nmigr == 0) {
      migrate(pop, tsp, rnd, mpi);
    }

    hist << gen << "," << pop.best_length() << "," << pop.half_mean_length()
         << "\n";
  }

  /* find global best and print it */
  double local_best = pop.best_length();
  if (mpi.rank() == rank_of_best(local_best, mpi)) {
    std::ofstream best(best_filename);
    const std::vector<std::size_t> &best_order = pop.best().order();
    for (auto i : best_order) {
      std::vector<double> pos = tsp.city(i).get_pos();
      for (std::size_t x = 0; x < pos.size() - 1; x++) {
        best << x << ",";
      }
      best << pos.back() << "\n";
    }
  }
}
```

### Ex. 10.2: Application to real data and usefulness of migrations
> ***goal**: run the optimization algorithm for the data on italian* capoluoghi di provincia *and compare the results in presence/absence of migrations*

Instead of generating citis randomly with
```cpp
  std::vector<City> cities = generate_and_bcast_cities(ndim, ncities, rnd, mpi);
```

we can read them from a *'input/capoluoghi.csv'* file with the `TSP::TSP(std::string filename)` constructor, then run the computation using `mpiexec -n 4 ./run_parallel`.

This is a comparison of 
- a run with a single migrant each process, once every 30 generations
- a run with no migrations at all (the if(..) migration loop was commented out)